In [2]:
import os
import json
from pathlib import Path
from tau2_enhanced.analysis.analyzer import LogAnalyzer
from tau2_enhanced.analysis.visualizer import LogVisualizer
from tau2_enhanced.logging.events import ToolExecutionEvent

SCRIPT_DIR = Path('.').parent.resolve()
PROJECT_ROOT = SCRIPT_DIR.parent
LOG_FILE = PROJECT_ROOT / "samples/logs/baseline_airline_xai_grok3_gemini2_5_flash_reduced.json"

2025-10-22 21:49:15.003 | INFO     | tau2.utils.utils:<module>:27 - Using data directory from source: /Users/jitraychowdhury/Documents/code_personal/ai_agent_analysis/tau2/tau2-bench/data
2025-10-22 21:49:16.126 | INFO     | tau2.utils.llm_utils:<module>:65 - LiteLLM: Cache is disabled
2025-10-22 21:49:16.126 | WARNING  | tau2.utils.llm_utils:<module>:72 - Sonnet thinking is disabled
2025-10-22 21:49:16.161 | DEBUG    | tau2.registry:<module>:174 - Registering default components...
2025-10-22 21:49:16.161 | DEBUG    | tau2.registry:<module>:194 - Default components registered successfully. Registry info: {
  "domains": [
    "mock",
    "airline",
    "retail",
    "telecom",
    "telecom-workflow"
  ],
  "agents": [
    "llm_agent",
    "llm_agent_gt",
    "llm_agent_solo"
  ],
  "users": [
    "user_simulator",
    "dummy_user"
  ],
  "task_sets": [
    "mock",
    "airline",
    "retail",
    "telecom_full",
    "telecom_small",
    "telecom",
    "telecom-workflow"
  ]
}
2025-10-22

In [3]:

def analyze_logs(log_file: Path):
    """
    Loads, analyzes, and visualizes execution logs from a file.
    """
    print(f"📁 Loading logs from: {log_file}")

    try:
        with log_file.open('r') as f:
            data = json.load(f)
        print(f"  ✅ Successfully loaded log file")
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"❌ Error loading log file: {e}")
        return

    # Detect log format and handle accordingly
    if 'execution_events' in data:
        # New simplified format: direct execution_events array
        print("  📊 Detected new simplified enhanced logs format")

        # Try to load the corresponding standard results file for simulation success data
        if '_enhanced_logs' in log_file.name:
            standard_file = Path(str(log_file).replace('_enhanced_logs', ''))
            try:
                with standard_file.open('r') as f:
                    standard_data = json.load(f)
                print(f"  ✅ Loaded standard results file for simulation data")

                # Add simulation data and tasks to the enhanced logs for analysis
                data['simulations'] = standard_data.get('simulations', [])
                data['tasks'] = standard_data.get('tasks', [])

            except (FileNotFoundError, json.JSONDecodeError) as e:
                print(f"  ⚠️  Could not load standard results file: {e}")

        analyzer_data = data  # Use combined data
    elif 'simulations' in data:
        # Legacy format: simulations with embedded logs
        print("  📊 Detected legacy enhanced logs format")
        analyzer_data = data
    else:
        print("❌ Unrecognized log format - no execution_events or simulations found")
        return

    # The LogAnalyzer is capable of handling different log formats.
    # We pass the entire loaded JSON data to it.
    analyzer = LogAnalyzer(analyzer_data)
    return analyzer 
analyzer = analyze_logs(LOG_FILE)

📁 Loading logs from: /Users/jitraychowdhury/Documents/code_personal/ai_agent_analysis/tau2/tau2-enhanced/samples/logs/baseline_airline_xai_grok3_gemini2_5_flash_reduced.json
  ✅ Successfully loaded log file
  📊 Detected legacy enhanced logs format


In [4]:
analyzer.get_summary_metrics()

{'total_simulations': 200,
 'successful_simulations': 115,
 'task_success_rate': 0.575,
 'total_trials': 4,
 'total_tasks': 50,
 'total_tool_calls': 1162,
 'successful_calls': 384,
 'failed_calls': 204,
 'tool_success_rate': 0.6530612244897959,
 'tool_error_rate': 0.34693877551020413,
 'total_execution_time': np.float64(0.31214165687561035),
 'average_execution_time': np.float64(0.00026862448956592976),
 'median_execution_time': np.float64(5.3882598876953125e-05),
 'state_changing_calls': 139,
 'read_only_calls': 1023,
 'most_common_tool': 'get_reservation_details',
 'slowest_tool_avg': 'get_user_details',
 'fastest_tool_avg': 'transfer_to_human_agents',
 'execution_timespan': 7286.293608,
 'tools_used': 13,
 'success_metric_source': 'action_checks'}

In [5]:
analyzer.get_tool_performance()

,tool_name,total_calls,successful_calls,avg_execution_time,median_execution_time,min_execution_time,max_execution_time,total_execution_time,state_changing_calls,avg_result_size,failed_calls,success_rate,error_rate,state_change_rate,performance_category
4,get_reservation_details,488,217,0.000101,0.000049,0.000027,0.023534,0.049372,0,200.000000,11.0,0.444672,0.022541,0.0,poor
6,search_direct_flight,164,17,0.000236,0.000225,0.000169,0.000932,0.038724,0,88.926829,63.0,0.103659,0.384146,0.0,poor
5,get_user_details,158,56,0.000823,0.000050,0.000037,0.067323,0.130068,0,200.000000,0.0,0.354430,0.000000,0.0,poor
7,search_onestop_flight,100,100,0.000681,0.000703,0.000170,0.002618,0.068130,0,120.800000,0.0,1.000000,0.000000,0.0,excellent
11,update_reservation_flights,62,38,0.000132,0.000120,0.000093,0.000755,0.008206,62,158.064516,46.0,0.612903,0.741935,1.0,poor
3,get_flight_status,56,56,0.000062,0.000054,0.000046,0.000407,0.003467,0,7.714286,0.0,1.000000,0.000000,0.0,excellent
9,transfer_to_human_agents,56,0,0.000060,0.000047,0.000038,0.000759,0.003381,0,19.000000,4.0,0.000000,0.071429,0.0,poor
2,cancel_reservation,39,29,0.000148,0.000129,0.000105,0.000533,0.005785,39,200.000000,22.0,0.743590,0.564103,1.0,poor
10,update_reservation_baggages,16,9,0.000079,0.000073,0.000060,0.000132,0.001264,16,200.000000,15.0,0.562500,0.937500,1.0,poor
0,book_reservation,9,5,0.000216,0.000215,0.000197,0.000249,0.001948,9,177.777778,28.0,0.555556,3.111111,1.0,poor


In [6]:
analyzer.get_failure_analysis()

,tool_name,error_category,count,simulations_affected,avg_action_reward,primary_failure_category,failure_rate,example_args
4,search_direct_flight,ActionCheckFailure,63,19,0.0,never_called,0.787500,"{'origin': 'BOS', 'destination': 'MCO', 'date'..."
8,update_reservation_flights,ActionCheckFailure,46,29,0.0,never_called,0.547619,"{'reservation_id': 'XEHM4B', 'cabin': 'economy..."
0,book_reservation,ActionCheckFailure,28,22,0.0,never_called,0.848485,"{'user_id': 'mohamed_silva_9265', 'origin': 'J..."
2,cancel_reservation,ActionCheckFailure,22,15,0.0,never_called,0.431373,{'reservation_id': 'XEHM4B'}
7,update_reservation_baggages,ActionCheckFailure,15,15,0.0,never_called,0.625000,"{'reservation_id': 'FQ8APE', 'total_baggages':..."
3,get_reservation_details,ActionCheckFailure,11,5,0.0,called_but_no_match,0.048246,{'reservation_id': 'SDZQKO'}
5,send_certificate,ActionCheckFailure,8,8,0.0,never_called,0.666667,"{'user_id': 'noah_muller_9847', 'amount': 50}"
1,calculate,ActionCheckFailure,4,4,0.0,never_called,1.000000,{'expression': '2 * ((350 - 122) + (499 - 127))'}
6,transfer_to_human_agents,ActionCheckFailure,4,4,0.0,never_called,1.000000,{'summary': 'User wants to change my upcoming ...
9,update_reservation_passengers,ActionCheckFailure,3,3,0.0,never_called,0.250000,"{'reservation_id': '3RK2T9', 'passengers': [{'..."


In [7]:
analyzer.get_state_change_analysis()

,state_changed,tool_name,total_calls,successful_calls,avg_execution_time,min_execution_time,max_execution_time,failed_calls,success_rate,error_rate,category,performance_rating
11,True,update_reservation_flights,62,38,0.000132,0.000093,0.000755,46.0,0.612903,0.741935,State-Changing,poor
8,True,cancel_reservation,39,29,0.000148,0.000105,0.000533,22.0,0.743590,0.564103,State-Changing,poor
10,True,update_reservation_baggages,16,9,0.000079,0.000060,0.000132,15.0,0.562500,0.937500,State-Changing,poor
7,True,book_reservation,9,5,0.000216,0.000197,0.000249,28.0,0.555556,3.111111,State-Changing,poor
12,True,update_reservation_passengers,9,9,0.000116,0.000097,0.000166,3.0,1.000000,0.333333,State-Changing,excellent
9,True,send_certificate,4,4,0.000162,0.000079,0.000408,8.0,1.000000,2.000000,State-Changing,excellent
2,False,get_reservation_details,488,217,0.000101,0.000027,0.023534,11.0,0.444672,0.022541,Read-Only,poor
4,False,search_direct_flight,164,17,0.000236,0.000169,0.000932,63.0,0.103659,0.384146,Read-Only,poor
3,False,get_user_details,158,56,0.000823,0.000037,0.067323,0.0,0.354430,0.000000,Read-Only,poor
5,False,search_onestop_flight,100,100,0.000681,0.000170,0.002618,0.0,1.000000,0.000000,Read-Only,excellent


In [8]:
analyzer.get_tool_sequence_analysis()

,source,target,count
0,get_reservation_details,get_reservation_details,287
1,get_user_details,get_reservation_details,134
2,search_direct_flight,search_onestop_flight,68
3,search_direct_flight,search_direct_flight,66
4,search_onestop_flight,search_direct_flight,45
...,...,...,...
62,get_flight_status,search_direct_flight,1
61,cancel_reservation,update_reservation_flights,1
60,update_reservation_flights,transfer_to_human_agents,1
59,update_reservation_baggages,cancel_reservation,1


In [9]:
analyzer.identify_bottlenecks(time_threshold=0.01)

,tool_name,tool_call_id,tool_args,timestamp,execution_time,success,error_message,requestor,result_preview,state_changed,...,optional_args_provided,missing_args,unexpected_args,result_type,result_complexity_score,result_contains_errors,result_truncated,has_result,datetime,error_details
712,get_user_details,get_user_details_0,{'user_id': 'ivan_muller_7015'},1.759137e+09,0.067323,True,None,assistant,user_id='ivan_muller_7015' name=Name(first_nam...,False,...,[],[],[],str,None,False,False,True,2025-09-29 09:07:59.039079905,None
757,get_user_details,get_user_details_0,{'user_id': 'daiki_lee_6144'},1.759137e+09,0.042887,True,None,assistant,user_id='daiki_lee_6144' name=Name(first_name=...,False,...,[],[],[],str,None,False,False,True,2025-09-29 09:12:38.203679085,None
235,get_reservation_details,get_reservation_details_4,{'reservation_id': 'SE9KEL'},1.759134e+09,0.023534,True,None,assistant,reservation_id='SE9KEL' user_id='sophia_martin...,False,...,[],[],[],str,None,False,False,True,2025-09-29 08:21:20.553643942,None
974,get_user_details,get_user_details_0,{'user_id': 'amelia_sanchez_4739'},1.759139e+09,0.010966,True,None,assistant,user_id='amelia_sanchez_4739' name=Name(first_...,False,...,[],[],[],str,None,False,False,True,2025-09-29 09:47:13.832218885,None


In [10]:
analyzer.get_temporal_analysis()

{'execution_velocity_per_minute': 9.568650915116953,
 'peak_activity_hour': np.int32(8),
 'quiet_activity_hour': np.int32(10),
 'hourly_patterns': {'call_count': {8: 637, 9: 480, 10: 45},
  'avg_execution_time': {8: 0.00016779914568507316,
   9: 0.0004222149650255839,
   10: 5.756484137641059e-05},
  'success_rate': {8: 0.9905808477237049, 9: 0.9833333333333333, 10: 1.0},
  'state_change_rate': {8: 0.13343799058084774,
   9: 0.10625,
   10: 0.06666666666666667}},
 'time_span_minutes': 121.4382268,
 'first_call': '2025-09-29T08:07:47.922207117',
 'last_call': '2025-09-29T10:09:14.215816021'}

In [11]:
analyzer.get_performance_trends()

{'success_rate_trend': 'improving',
 'execution_speed_trend': 'improving',
 'bucket_analysis': {'success_rate': {0: 0.9829059829059829,
   1: 1.0,
   2: 1.0,
   3: 0.9655172413793104,
   4: 1.0,
   5: 0.9741379310344828,
   6: 0.9913793103448276,
   7: 0.9913793103448276,
   8: 0.9741379310344828,
   9: 1.0},
  'avg_execution_time': {0: 0.0001475546095106337,
   1: 0.00010401010513305664,
   2: 0.00028978339556989997,
   3: 0.00018482783745075096,
   4: 0.00010860993944365403,
   5: 0.00019662133578596445,
   6: 0.0010777321355096225,
   7: 0.00012745117319041285,
   8: 0.0003107334005421606,
   9: 0.00014106432596842447},
  'call_count': {0: 117,
   1: 116,
   2: 116,
   3: 116,
   4: 116,
   5: 116,
   6: 116,
   7: 116,
   8: 116,
   9: 117},
  'state_change_rate': {0: 0.1452991452991453,
   1: 0.15517241379310345,
   2: 0.11206896551724138,
   3: 0.1896551724137931,
   4: 0.11206896551724138,
   5: 0.08620689655172414,
   6: 0.11206896551724138,
   7: 0.06896551724137931,
   8: 0.1

In [12]:
analyzer.get_tool_usage_patterns()

{'total_unique_tools': 13,
 'most_used_tool': {'name': 'get_reservation_details',
  'calls': np.int64(488),
  'percentage': np.float64(41.996557659208264)},
 'least_used_tool': {'name': 'calculate',
  'calls': np.int64(1),
  'percentage': np.float64(0.08605851979345956)},
 'tool_usage_distribution': {'get_reservation_details': 488,
  'search_direct_flight': 164,
  'get_user_details': 158,
  'search_onestop_flight': 100,
  'update_reservation_flights': 62,
  'transfer_to_human_agents': 56,
  'get_flight_status': 56,
  'cancel_reservation': 39,
  'update_reservation_baggages': 16,
  'book_reservation': 9,
  'update_reservation_passengers': 9,
  'send_certificate': 4,
  'calculate': 1},
 'diversity_index': np.float64(1.845577532194016),
 'usage_concentration': 'distributed'}

In [13]:
analyzer.get_error_pattern_analysis()

{'total_errors': 14,
 'error_rate': 0.012048192771084338,
 'tools_with_errors': 2,
 'error_types': {},
 'most_common_error': None,
 'avg_error_execution_time': np.float64(0.00011995860508510045),
 'avg_success_execution_time': np.float64(0.00027043748815715933),
 'error_time_correlation': 'shorter',
 'tools_by_error_rate': {'update_reservation_flights': 0.20967741935483872,
  'book_reservation': 0.1111111111111111,
  'calculate': nan,
  'cancel_reservation': nan,
  'get_flight_status': nan,
  'get_reservation_details': nan,
  'get_user_details': nan,
  'search_direct_flight': nan,
  'search_onestop_flight': nan,
  'send_certificate': nan,
  'transfer_to_human_agents': nan,
  'update_reservation_baggages': nan,
  'update_reservation_passengers': nan}}

In [14]:
analyzer.get_requestor_analysis()

{'requestor_breakdown': {'assistant': {'total_calls': 1162,
   'success_rate': np.float64(0.9879518072289156),
   'avg_execution_time': np.float64(0.00026862448956592976),
   'state_change_rate': np.float64(0.11962134251290878),
   'most_used_tools': {'get_reservation_details': 488,
    'search_direct_flight': 164,
    'get_user_details': 158},
   'error_rate': np.float64(0.012048192771084338)}},
 'total_requestors': 1}

In [15]:
analyzer.get_advanced_statistics()

{'execution_time_stats': {'mean': np.float64(0.00026862448956592976),
  'median': np.float64(5.3882598876953125e-05),
  'std': np.float64(0.002463581980206369),
  'min': np.float64(2.7418136596679688e-05),
  'max': np.float64(0.0673227310180664),
  'p25': np.float64(4.8160552978515625e-05),
  'p75': np.float64(0.00017017126083374023),
  'p95': np.float64(0.00070185661315918),
  'p99': np.float64(0.0013212800025939944)},
 'success_rate_confidence': {'rate': np.float64(0.9879518072289156),
  'lower_bound': np.float64(0.9816787125671287),
  'upper_bound': np.float64(0.9942249018907026),
  'sample_size': 1162},
 'tool_distribution': {'entropy': np.float64(1.845577532194016),
  'gini_coefficient': np.float64(0.6222692969680922),
  'concentration_ratio': np.float64(0.4199655765920826)}}

In [16]:
analyzer.get_argument_analysis()

{'argument_statistics': {'total_calls': 1162,
  'calls_with_arguments': 1162,
  'calls_without_arguments': 0,
  'avg_args_per_call': np.float64(1.7925989672977625),
  'max_args_in_call': np.int64(11),
  'min_args_in_call': np.int64(1)},
 'complexity_analysis': {'avg_complexity': np.float64(0.2043509977409639),
  'max_complexity': np.float64(1.0),
  'high_complexity_calls': 9,
  'low_complexity_calls': 799,
  'complexity_distribution': {'low (0-0.3)': 799,
   'medium (0.3-0.7)': 354,
   'high (0.7-1.0)': 9}},
 'type_analysis': {'most_common_types': {'str': 1931, 'list': 98, 'int': 54},
  'total_unique_types': 3,
  'type_distribution': {'str': 1931, 'list': 98, 'int': 54}},
 'size_analysis': {'avg_args_size_bytes': np.float64(70.21084337349397),
  'max_args_size_bytes': np.int64(915),
  'min_args_size_bytes': np.int64(26),
  'large_args_calls': 0,
  'size_distribution': {'small (<1KB)': 1162,
   'medium (1KB-10KB)': 0,
   'large (>10KB)': 0}},
 'security_analysis': {'calls_with_sensitive

In [17]:
analyzer.get_argument_correlation_analysis()

{'correlations': {'args_count_vs_execution_time': {'correlation': np.float64(0.005158379859283713),
   'interpretation': 'Very weak positive correlation'},
  'args_size_vs_execution_time': {'correlation': np.float64(-0.018413091230688065),
   'interpretation': 'Very weak negative correlation'},
  'complexity_vs_success_rate': {'correlation': np.float64(-0.3190991566329591),
   'interpretation': 'Moderate negative correlation'}},
 'performance_comparisons': {}}

In [18]:
analyzer.get_result_analysis()

{'result_statistics': {'total_calls': 1162,
  'calls_with_results': 1148,
  'calls_without_results': 14,
  'result_rate': 0.9879518072289156},
 'type_analysis': {'most_common_types': {'str': 1148},
  'total_unique_types': 1,
  'type_distribution': {'str': 1148}},
 'size_analysis': {'avg_result_size': np.float64(156.52409638554218),
  'max_result_size': np.int64(200),
  'min_result_size': np.int64(0),
  'median_result_size': np.float64(200.0),
  'size_distribution': {'small (<1KB)': 1162,
   'medium (1KB-100KB)': 0,
   'large (>100KB)': 0}},
 'quality_analysis': {'results_with_errors': 0,
  'error_result_rate': 0.0,
  'truncated_results': 0,
  'truncation_rate': 0.0}}

In [19]:
analyzer.get_detailed_failure_breakdown()

,tool_name,error_category,simulation_id,task_id,trial,action_reward,arguments,failure_category,similarity,expected_args,actual_args,has_match
0,update_reservation_flights,ActionCheckFailure,7_0,7,0,0.0,"{'reservation_id': 'XEHM4B', 'cabin': 'economy...",never_called,0.0,"{'reservation_id': 'XEHM4B', 'cabin': 'economy...",{},False
1,cancel_reservation,ActionCheckFailure,7_0,7,0,0.0,{'reservation_id': 'XEHM4B'},called_but_no_match,0.0,{'reservation_id': 'XEHM4B'},{},False
2,transfer_to_human_agents,ActionCheckFailure,13_0,13,0,0.0,{'summary': 'User wants to change my upcoming ...,never_called,0.0,{'summary': 'User wants to change my upcoming ...,{},False
3,search_direct_flight,ActionCheckFailure,12_0,12,0,0.0,"{'origin': 'BOS', 'destination': 'MCO', 'date'...",never_called,0.0,"{'origin': 'BOS', 'destination': 'MCO', 'date'...",{},False
4,search_direct_flight,ActionCheckFailure,12_0,12,0,0.0,"{'origin': 'MCO', 'destination': 'MSP', 'date'...",never_called,0.0,"{'origin': 'MCO', 'destination': 'MSP', 'date'...",{},False
...,...,...,...,...,...,...,...,...,...,...,...,...
199,search_direct_flight,ActionCheckFailure,44_3,44,3,0.0,"{'origin': 'ORD', 'destination': 'PHL', 'date'...",never_called,0.0,"{'origin': 'ORD', 'destination': 'PHL', 'date'...",{},False
200,cancel_reservation,ActionCheckFailure,44_3,44,3,0.0,{'reservation_id': 'S61CZX'},never_called,0.0,{'reservation_id': 'S61CZX'},{},False
201,update_reservation_flights,ActionCheckFailure,44_3,44,3,0.0,"{'reservation_id': 'NM1VX1', 'cabin': 'busines...",never_called,0.0,"{'reservation_id': 'NM1VX1', 'cabin': 'busines...",{},False
202,update_reservation_flights,ActionCheckFailure,44_3,44,3,0.0,"{'reservation_id': 'H8Q05L', 'cabin': 'busines...",never_called,0.0,"{'reservation_id': 'H8Q05L', 'cabin': 'busines...",{},False
